In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
# Define dataset directory
DATA_PATH = "../data/raw/"

# Load datasets
customers = pd.read_csv(DATA_PATH + "olist_customers_dataset.csv")
orders = pd.read_csv(DATA_PATH + "olist_orders_dataset.csv")
order_items = pd.read_csv(DATA_PATH + "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA_PATH + "olist_order_payments_dataset.csv")
reviews = pd.read_csv(DATA_PATH + "olist_order_reviews_dataset.csv")
products = pd.read_csv(DATA_PATH + "olist_products_dataset.csv")
sellers = pd.read_csv(DATA_PATH + "olist_sellers_dataset.csv")
geolocation = pd.read_csv(DATA_PATH + "olist_geolocation_dataset.csv")
category_translation = pd.read_csv(DATA_PATH + "product_category_name_translation.csv")

print("All datasets loaded successfully!")

All datasets loaded successfully!


In [3]:
# Create dictionary of datasets
datasets = {
    "Customers": customers,
    "Orders": orders,
    "Order Items": order_items,
    "Payments": payments,
    "Reviews": reviews,
    "Products": products,
    "Sellers": sellers,
    "Geolocation": geolocation,
    "Category Translation": category_translation
}

# Check dataset dimensions
for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")

Customers: 99,441 rows, 5 columns
Orders: 99,441 rows, 8 columns
Order Items: 112,650 rows, 7 columns
Payments: 103,886 rows, 5 columns
Reviews: 99,224 rows, 7 columns
Products: 32,951 rows, 9 columns
Sellers: 3,095 rows, 4 columns
Geolocation: 1,000,163 rows, 5 columns
Category Translation: 71 rows, 2 columns


In [7]:
# Check missing values in all datasets
for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 40)

    missing = df.isnull().sum()
    missing_percentage = (missing / len(df)) * 100

    missing_summary = pd.DataFrame({
        "Missing Values": missing,
        "Percentage (%)": missing_percentage
    })

    print(missing_summary[missing_summary["Missing Values"] > 0])


Customers
----------------------------------------
Empty DataFrame
Columns: [Missing Values, Percentage (%)]
Index: []

Orders
----------------------------------------
                               Missing Values  Percentage (%)
order_approved_at                         160        0.160899
order_delivered_carrier_date             1783        1.793023
order_delivered_customer_date            2965        2.981668

Order Items
----------------------------------------
Empty DataFrame
Columns: [Missing Values, Percentage (%)]
Index: []

Payments
----------------------------------------
Empty DataFrame
Columns: [Missing Values, Percentage (%)]
Index: []

Reviews
----------------------------------------
                        Missing Values  Percentage (%)
review_comment_title             87656       88.341530
review_comment_message           58247       58.702532

Products
----------------------------------------
                            Missing Values  Percentage (%)
product_category_

In [8]:
# Create a complete missing values summary
missing_data = []

for name, df in datasets.items():
    for column in df.columns:
        missing_count = df[column].isnull().sum()

        if missing_count > 0:
            missing_data.append({
                "Dataset": name,
                "Column": column,
                "Missing Values": missing_count,
                "Percentage (%)": round(
                    (missing_count / len(df)) * 100, 2
                )
            })

missing_summary = pd.DataFrame(missing_data)

missing_summary

,Dataset,Column,Missing Values,Percentage (%)
0,Orders,order_approved_at,160,0.16
1,Orders,order_delivered_carrier_date,1783,1.79
2,Orders,order_delivered_customer_date,2965,2.98
3,Reviews,review_comment_title,87656,88.34
4,Reviews,review_comment_message,58247,58.70
5,Products,product_category_name,610,1.85
6,Products,product_name_lenght,610,1.85
7,Products,product_description_lenght,610,1.85
8,Products,product_photos_qty,610,1.85
9,Products,product_weight_g,2,0.01


In [9]:
# Analyze missing values based on order status

orders_missing_analysis = orders.groupby("order_status").agg(
    total_orders=("order_id", "count"),
    missing_approved=("order_approved_at", lambda x: x.isnull().sum()),
    missing_carrier=("order_delivered_carrier_date", lambda x: x.isnull().sum()),
    missing_delivered=("order_delivered_customer_date", lambda x: x.isnull().sum())
)

orders_missing_analysis

,total_orders,missing_approved,missing_carrier,missing_delivered
order_status,,,,
approved,2,0,2,2
canceled,625,141,550,619
created,5,5,5,5
delivered,96478,14,2,8
invoiced,314,0,314,314
processing,301,0,301,301
shipped,1107,0,0,1107
unavailable,609,0,609,609


In [10]:
# Identify delivered orders with missing delivery dates

delivered_missing = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isnull())
]

delivered_missing[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
]

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


In [11]:
# Identify delivered orders with missing carrier dates

delivered_missing_carrier = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_carrier_date"].isnull())
]

delivered_missing_carrier[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
]

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
73222,2aa91108853cecb43c84a5dc5b277475,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaN,2017-11-20 19:44:47,2017-11-14 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00


In [12]:
# Analyze missing values in review comments

reviews_missing_analysis = pd.DataFrame({
    "Total Reviews": len(reviews),
    "Missing Title": reviews["review_comment_title"].isnull().sum(),
    "Missing Message": reviews["review_comment_message"].isnull().sum(),
    "Available Title": reviews["review_comment_title"].notnull().sum(),
    "Available Message": reviews["review_comment_message"].notnull().sum()
}, index=["Review Analysis"])

reviews_missing_analysis.T

,Review Analysis
Total Reviews,99224
Missing Title,87656
Missing Message,58247
Available Title,11568
Available Message,40977


In [13]:
# Analyze missing values in products

products_missing_analysis = pd.DataFrame({
    "Missing Values": products.isnull().sum(),
    "Percentage (%)": (products.isnull().sum() / len(products) * 100).round(2)
})

products_missing_analysis[
    products_missing_analysis["Missing Values"] > 0
]

,Missing Values,Percentage (%)
product_category_name,610,1.85
product_name_lenght,610,1.85
product_description_lenght,610,1.85
product_photos_qty,610,1.85
product_weight_g,2,0.01
product_length_cm,2,0.01
product_height_cm,2,0.01
product_width_cm,2,0.01


In [14]:
# Check products with missing category information

missing_category = products[
    products["product_category_name"].isnull()
]

print("Products with missing category information:")
print("Total rows:", len(missing_category))

missing_category[
    [
        "product_id",
        "product_category_name",
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty"
    ]
].head(10)


Products with missing category information:
Total rows: 610


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN
244,e10758160da97891c2fdcbc35f0f031d,NaN,NaN,NaN,NaN
294,39e3b9b12cd0bf8ee681bbc1c130feb5,NaN,NaN,NaN,NaN
299,794de06c32a626a5692ff50e4985d36f,NaN,NaN,NaN,NaN
347,7af3e2da474486a3519b0cba9dea8ad9,NaN,NaN,NaN,NaN
428,629beb8e7317703dcc5f35b5463fd20e,NaN,NaN,NaN,NaN


In [15]:
# Check products with missing physical attributes

missing_dimensions = products[
    products["product_weight_g"].isnull()
]

print("Products with missing physical attributes:")
print("Total rows:", len(missing_dimensions))

missing_dimensions[
    [
        "product_id",
        "product_category_name",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
]

Products with missing physical attributes:
Total rows: 2


,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN


In [16]:
# Check duplicate rows in all datasets

duplicate_summary = []

for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()

    duplicate_summary.append({
        "Dataset": name,
        "Duplicate Rows": duplicate_count,
        "Percentage (%)": round(
            (duplicate_count / len(df)) * 100, 2
        )
    })

duplicate_summary = pd.DataFrame(duplicate_summary)

duplicate_summary

,Dataset,Duplicate Rows,Percentage (%)
0,Customers,0,0.00
1,Orders,0,0.00
2,Order Items,0,0.00
3,Payments,0,0.00
4,Reviews,0,0.00
5,Products,0,0.00
6,Sellers,0,0.00
7,Geolocation,261831,26.18
8,Category Translation,0,0.00


In [17]:
# Display sample duplicated geolocation records

geolocation_duplicates = geolocation[
    geolocation.duplicated(keep=False)
]

print("Total duplicated rows:", len(geolocation_duplicates))

geolocation_duplicates.head(10)

Total duplicated rows: 390005


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
6,1047,-23.546273,-46.641225,sao paulo,SP
7,1013,-23.546923,-46.634264,sao paulo,SP
8,1029,-23.543769,-46.634278,sao paulo,SP
9,1011,-23.547640,-46.636032,sao paulo,SP
10,1013,-23.547325,-46.634184,sao paulo,SP
13,1012,-23.548946,-46.634671,sao paulo,SP
15,1046,-23.546081,-46.644820,sao paulo,SP


In [18]:
# Check exact duplicate geolocation records

exact_duplicates = geolocation[
    geolocation.duplicated(keep=False)
]

sample_duplicate = exact_duplicates[
    exact_duplicates["geolocation_zip_code_prefix"] == 1013
]

sample_duplicate

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
7,1013,-23.546923,-46.634264,sao paulo,SP
10,1013,-23.547325,-46.634184,sao paulo,SP
91,1013,-23.547325,-46.634184,sao paulo,SP
167,1013,-23.547308,-46.634251,sao paulo,SP
198,1013,-23.546421,-46.634362,sao paulo,SP
297,1013,-23.547686,-46.634140,sao paulo,SP
449,1013,-23.546421,-46.634362,sao paulo,SP
471,1013,-23.546421,-46.634362,sao paulo,SP
599,1013,-23.547325,-46.634184,sao paulo,SP
836,1013,-23.547686,-46.634140,sao paulo,SP


In [19]:
# Remove exact duplicate rows from geolocation

geolocation_cleaned = geolocation.drop_duplicates().copy()

print("Original rows:", len(geolocation))
print("Rows after removing duplicates:", len(geolocation_cleaned))
print("Removed duplicates:", len(geolocation) - len(geolocation_cleaned))

Original rows: 1000163
Rows after removing duplicates: 738332
Removed duplicates: 261831


In [20]:
# Check data types of all datasets

for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 50)
    print(df.dtypes)


Customers
--------------------------------------------------
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

Orders
--------------------------------------------------
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

Order Items
--------------------------------------------------
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

Payments
--------------------------------------------------
order_id                    s

In [21]:
# Convert order date columns to datetime

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders[column] = pd.to_datetime(orders[column], errors="coerce")

# Check converted data types
orders[date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [22]:
# Validate missing values after datetime conversion

for column in date_columns:
    print(
        f"{column}: {orders[column].isnull().sum()} missing values"
    )

order_purchase_timestamp: 0 missing values
order_approved_at: 160 missing values
order_delivered_carrier_date: 1783 missing values
order_delivered_customer_date: 2965 missing values
order_estimated_delivery_date: 0 missing values


In [23]:
# Check numeric data types

numeric_columns = {
    "Order Items": order_items,
    "Payments": payments,
    "Reviews": reviews,
    "Products": products
}

for name, df in numeric_columns.items():
    print(f"\n{name}")
    print("-" * 40)
    print(df.select_dtypes(include="number").dtypes)


Order Items
----------------------------------------
order_item_id      int64
price            float64
freight_value    float64
dtype: object

Payments
----------------------------------------
payment_sequential        int64
payment_installments      int64
payment_value           float64
dtype: object

Reviews
----------------------------------------
review_score    int64
dtype: object

Products
----------------------------------------
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object


In [24]:
# Check invalid values in order items

print("Price Analysis")
print("Minimum price:", order_items["price"].min())
print("Zero price:", (order_items["price"] == 0).sum())
print("Negative price:", (order_items["price"] < 0).sum())

print("\nFreight Value Analysis")
print("Minimum freight:", order_items["freight_value"].min())
print("Zero freight:", (order_items["freight_value"] == 0).sum())
print("Negative freight:", (order_items["freight_value"] < 0).sum())

Price Analysis
Minimum price: 0.85
Zero price: 0
Negative price: 0

Freight Value Analysis
Minimum freight: 0.0
Zero freight: 383
Negative freight: 0


In [25]:
# Inspect records with zero freight value

zero_freight = order_items[
    order_items["freight_value"] == 0
]

print("Total zero freight records:", len(zero_freight))

display(
    zero_freight[
        ["order_id", "order_item_id", "product_id",
         "seller_id", "price", "freight_value"]
    ].head(10)
)

Total zero freight records: 383


,order_id,order_item_id,product_id,seller_id,price,freight_value
114,00404fa7a687c8c44ca69d42695aae73,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,99.9,0.0
258,00a870c6c06346e85335524935c600c0,1,aca2eb7d00ea1a7b8ebd4e68314663af,955fee9216a65b617aa5c0531780ce60,69.9,0.0
483,011c899816ea29773525bd3322dbb6aa,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,99.9,0.0
508,012b3f6ab7776a8ab3443a4ad7bef2e6,1,422879e10f46682990de24d770e7f83d,1f50f920176fa81dab994f9023523100,53.9,0.0
509,012b3f6ab7776a8ab3443a4ad7bef2e6,2,422879e10f46682990de24d770e7f83d,1f50f920176fa81dab994f9023523100,53.9,0.0
1784,04105b54650921ca3246f52e6f175f46,1,aca2eb7d00ea1a7b8ebd4e68314663af,955fee9216a65b617aa5c0531780ce60,69.9,0.0
2232,0517a3e68dac3308995edca2144db36e,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,99.9,0.0
2714,061ba2e2d7544790b6ed6b5b4dd9278c,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,110.0,0.0
3224,07441f525824bc6b31d4dc19c5d49fc9,1,53b36df67ebb7c41585e8d54d6772e08,4869f7a5dfa277a7dca6462dcf3b52b2,106.9,0.0
3378,079f16689c29acb6cab92978e6af2137,1,aca2eb7d00ea1a7b8ebd4e68314663af,955fee9216a65b617aa5c0531780ce60,69.9,0.0


In [26]:
# Analyze zero freight by seller

zero_freight_analysis = (
    zero_freight
    .groupby("seller_id")
    .agg(
        total_items=("order_id", "count"),
        total_orders=("order_id", "nunique"),
        total_products=("product_id", "nunique")
    )
    .sort_values("total_items", ascending=False)
)

print("Number of sellers with zero freight:",
      zero_freight["seller_id"].nunique())

display(zero_freight_analysis.head(10))

Number of sellers with zero freight: 9


,total_items,total_orders,total_products
seller_id,,,
7d13fca15225358621be4086e1eb0964,158,146,1
955fee9216a65b617aa5c0531780ce60,99,83,2
1f50f920176fa81dab994f9023523100,56,40,1
4869f7a5dfa277a7dca6462dcf3b52b2,56,56,2
37be5a7c751166fbc5f8ccba4119e043,9,9,1
c826c40d7b19f62a09e2d7c5e7295ee2,2,2,1
8581055ce74af1daba164fdbd55a40de,1,1,1
bc2ac6b95e1accce9858528ee566c17e,1,1,1
cc419e0650a3c5ba77189a1882b7556a,1,1,1


In [27]:
# Check invalid values in payments

print("Payment Value Analysis")
print("Minimum payment:", payments["payment_value"].min())
print("Zero payment:", (payments["payment_value"] == 0).sum())
print("Negative payment:", (payments["payment_value"] < 0).sum())

print("\nPayment Installments Analysis")
print("Minimum installments:", payments["payment_installments"].min())
print("Zero installments:", (payments["payment_installments"] == 0).sum())
print("Negative installments:", (payments["payment_installments"] < 0).sum())

Payment Value Analysis
Minimum payment: 0.0
Zero payment: 9
Negative payment: 0

Payment Installments Analysis
Minimum installments: 0
Zero installments: 2
Negative installments: 0


In [28]:
# Inspect zero payment records

zero_payment = payments[
    payments["payment_value"] == 0
]

print("Total zero payment records:", len(zero_payment))

display(zero_payment)

Total zero payment records: 9


,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


In [29]:
# Inspect zero payment installments

zero_installments = payments[
    payments["payment_installments"] == 0
]

print("Total zero installment records:", len(zero_installments))

display(zero_installments)

Total zero installment records: 2


,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [30]:
# Inspect all payment records from affected orders

affected_orders = zero_installments["order_id"].unique()

display(
    payments[
        payments["order_id"].isin(affected_orders)
    ].sort_values(
        ["order_id", "payment_sequential"]
    )
)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69


In [31]:
# Check invalid values in reviews

print("Review Score Analysis")
print("Minimum review score:", reviews["review_score"].min())
print("Maximum review score:", reviews["review_score"].max())

print("Zero review score:",
      (reviews["review_score"] == 0).sum())

print("Negative review score:",
      (reviews["review_score"] < 0).sum())

print("Review score above 5:",
      (reviews["review_score"] > 5).sum())

print("\nReview Score Distribution")
display(reviews["review_score"].value_counts().sort_index())

Review Score Analysis
Minimum review score: 1
Maximum review score: 5
Zero review score: 0
Negative review score: 0
Review score above 5: 0

Review Score Distribution


review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [32]:
# Check invalid values in product attributes

numeric_product_columns = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for column in numeric_product_columns:
    print(f"\n{column}")
    print("Minimum:", products[column].min())
    print("Zero values:", (products[column] == 0).sum())
    print("Negative values:", (products[column] < 0).sum())


product_name_lenght
Minimum: 5.0
Zero values: 0
Negative values: 0

product_description_lenght
Minimum: 4.0
Zero values: 0
Negative values: 0

product_photos_qty
Minimum: 1.0
Zero values: 0
Negative values: 0

product_weight_g
Minimum: 0.0
Zero values: 4
Negative values: 0

product_length_cm
Minimum: 7.0
Zero values: 0
Negative values: 0

product_height_cm
Minimum: 2.0
Zero values: 0
Negative values: 0

product_width_cm
Minimum: 6.0
Zero values: 0
Negative values: 0


In [33]:
# Inspect products with zero weight

zero_weight = products[
    products["product_weight_g"] == 0
]

print("Total products with zero weight:", len(zero_weight))

display(zero_weight)

Total products with zero weight: 4


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51.0,529.0,1.0,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48.0,528.0,1.0,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0


In [34]:
# Complete product numeric validation

for column in numeric_product_columns:
    print(
        f"{column}: "
        f"Missing = {products[column].isnull().sum()}, "
        f"Zero = {(products[column] == 0).sum()}, "
        f"Negative = {(products[column] < 0).sum()}"
    )

product_name_lenght: Missing = 610, Zero = 0, Negative = 0
product_description_lenght: Missing = 610, Zero = 0, Negative = 0
product_photos_qty: Missing = 610, Zero = 0, Negative = 0
product_weight_g: Missing = 2, Zero = 4, Negative = 0
product_length_cm: Missing = 2, Zero = 0, Negative = 0
product_height_cm: Missing = 2, Zero = 0, Negative = 0
product_width_cm: Missing = 2, Zero = 0, Negative = 0


In [35]:
# Create cleaned dataset copies

customers_cleaned = customers.copy()
orders_cleaned = orders.copy()
order_items_cleaned = order_items.copy()
payments_cleaned = payments.copy()
reviews_cleaned = reviews.copy()
products_cleaned = products.copy()
sellers_cleaned = sellers.copy()
category_translation_cleaned = category_translation.copy()

# Use geolocation with exact duplicate rows removed
geolocation_cleaned = geolocation.drop_duplicates().copy()

print("Cleaned datasets are ready.")

Cleaned datasets are ready.


In [36]:
# Save cleaned datasets

OUTPUT_PATH = "../data/processed/"

cleaned_datasets = {
    "customers": customers_cleaned,
    "orders": orders_cleaned,
    "order_items": order_items_cleaned,
    "payments": payments_cleaned,
    "reviews": reviews_cleaned,
    "products": products_cleaned,
    "sellers": sellers_cleaned,
    "geolocation": geolocation_cleaned,
    "category_translation": category_translation_cleaned
}

for name, df in cleaned_datasets.items():
    df.to_csv(
        OUTPUT_PATH + f"{name}_cleaned.csv",
        index=False
    )

    print(f"{name}_cleaned.csv saved | Rows: {len(df):,}")

customers_cleaned.csv saved | Rows: 99,441
orders_cleaned.csv saved | Rows: 99,441
order_items_cleaned.csv saved | Rows: 112,650
payments_cleaned.csv saved | Rows: 103,886
reviews_cleaned.csv saved | Rows: 99,224
products_cleaned.csv saved | Rows: 32,951
sellers_cleaned.csv saved | Rows: 3,095
geolocation_cleaned.csv saved | Rows: 738,332
category_translation_cleaned.csv saved | Rows: 71


In [37]:
# Validate saved cleaned datasets

for name, original_df in cleaned_datasets.items():

    file_path = OUTPUT_PATH + f"{name}_cleaned.csv"

    df_check = pd.read_csv(file_path)

    print(
        f"{name}: "
        f"Original = {len(original_df):,} | "
        f"Saved = {len(df_check):,} | "
        f"Match = {len(original_df) == len(df_check)}"
    )

customers: Original = 99,441 | Saved = 99,441 | Match = True
orders: Original = 99,441 | Saved = 99,441 | Match = True
order_items: Original = 112,650 | Saved = 112,650 | Match = True
payments: Original = 103,886 | Saved = 103,886 | Match = True
reviews: Original = 99,224 | Saved = 99,224 | Match = True
products: Original = 32,951 | Saved = 32,951 | Match = True
sellers: Original = 3,095 | Saved = 3,095 | Match = True
geolocation: Original = 738,332 | Saved = 738,332 | Match = True
category_translation: Original = 71 | Saved = 71 | Match = True
